# 06 · Access sensitivity runs (MLflow)

Four MLflow runs per job run (#33), so the team can see when a data refresh shifts the numbers:
- the 10, 15 and 20 min limits on the walking network (`gold.block_access`, `gold.access_sensitivity`)
- one step-free run: seniors at 1.0 m/s and 15 min on `silver.dist_matrix_step_free`

Each run logs the paces, the limit and the data download dates as params, and seniors cut off, centres excluded and the T4 count as metrics. The app never reads MLflow. Overload sensitivity (eating share, distance decay) is Spec 2's.

In [0]:
%pip install "mlflow-skinny[databricks]>=2.15"

In [0]:
import mlflow
from pyspark.sql import functions as F
from tiers import load_assumptions, load_paces, reach_m, tier_expr  # tiers.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("silver_schema", "silver")  # dev_<name> for test runs
dbutils.widgets.text("gold_schema", "gold")      # dev_<name>_gold for test runs
dbutils.widgets.text("experiment", "")           # default: hawkerreach_access_sensitivity in your workspace folder
dbutils.widgets.text("job_run_id", "")           # the job passes {{job.run_id}}, so the four runs can be grouped
SILVER= f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
GOLD= f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"
user= spark.sql("SELECT current_user()").first()[0]
EXPERIMENT= dbutils.widgets.get("experiment") or f"/Users/{user}/hawkerreach_access_sensitivity"
JOB_RUN_ID= dbutils.widgets.get("job_run_id") or "manual"
mlflow.set_experiment(EXPERIMENT)
print("reading", SILVER, "and", GOLD, "| experiment", EXPERIMENT, "| job run", JOB_RUN_ID)

A= load_assumptions()
PACES= load_paces(A)                       # walk.pace.senior / typical / brisk
LIMIT_MIN= A["walk.limit_min"]["value"]    # 15
CUTOFF_M= A["walk.cutoff_m"]["value"]      # 2000
SENS_LIMITS= sorted({LIMIT_MIN, *(int(x) for x in str(A["access.sensitivity_limits_min"]["value"]).split(","))})

# data download dates: each bronze table's downloaded_at, and the OpenStreetMap file 03 saved
downloaded= {}
for t in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect():
    df= spark.table(f"{CATALOG}.bronze.{t.tableName}")
    if "downloaded_at" in df.columns:
        downloaded[f"downloaded.{t.tableName}"]= str(df.agg(F.max("downloaded_at")).first()[0])
osm= sorted(f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/bronze/raw_files") if f.name.startswith("osm_"))
downloaded["downloaded.osm"]= osm[-1].removesuffix(".osm.pbf").rsplit("_", 1)[-1] if osm else "none"

base_params= {**{f"pace.{k}": v for k, v in PACES.items()}, "cutoff_m": CUTOFF_M, "job_run_id": JOB_RUN_ID, **downloaded}
print(base_params)

In [0]:
blocks= spark.table(f"{GOLD}.block_access").select("block_id", "est_seniors", "tier_now", "nearest_walk_m")
centres= spark.table(f"{GOLD}.centres")
n_centres= centres.count()

def centres_excluded(matrix):
    """Operating centres no HDB block can reach within the cutoff on this network, as 04_gold_access counts them."""
    return centres.join(spark.table(matrix).select("centre_id").distinct(), "centre_id", "left_anti").count()

def tier_counts(walks, limit_min):
    """walks: block_id, est_seniors, walk_m (nearest; null if none in range). T4 blocks and seniors without a senior-pace walk."""
    t= walks.withColumn("tier", tier_expr(F.col("walk_m"), limit_min, PACES))
    r= t.agg(F.sum(F.when(F.col("tier") == "T4", 1).otherwise(0)).alias("t4"),
             F.sum(F.when(F.col("tier") != "T1", F.col("est_seniors")).otherwise(0)).alias("seniors_without")).first()
    return int(r["t4"] or 0), float(r["seniors_without"] or 0)

def log_run(name, params, metrics):
    with mlflow.start_run(run_name=name):
        mlflow.set_tags({"job_run_id": JOB_RUN_ID, "notebook": "06_sensitivity_mlflow"})
        mlflow.log_params({**base_params, **params})
        mlflow.log_metrics(metrics)
    print(name, metrics)
    return metrics

In [0]:
# 10, 15 and 20 min on the walking network
excluded= centres_excluded(f"{SILVER}.dist_matrix")
sens= spark.table(f"{GOLD}.access_sensitivity")
walks= blocks.select("block_id", "est_seniors", F.col("nearest_walk_m").alias("walk_m"))
results= {}
for limit in SENS_LIMITS:
    s= sens.filter(F.col("limit_min") == limit).agg(F.sum("seniors_cut_off").alias("total"), F.max("seniors_cut_off").alias("worst")).first()
    t4, without= tier_counts(walks, limit)
    results[limit]= log_run(f"limit_{limit}min", {"network": "walking", "limit_min": limit, "reach_m": [round(r) for r in reach_m(limit, PACES)]}, {
        "seniors_cut_off_total": float(s["total"] or 0),  # summed over every closable centre closing alone
        "seniors_cut_off_worst_centre": float(s["worst"] or 0),
        "seniors_without_walk": without,  # today, all centres open: seniors with no centre within the limit at 1.0 m/s
        "t4_blocks": t4,
        "centres_excluded": excluded,
    })

In [0]:
# step-free: seniors at 1.0 m/s and 15 min when steps can't be used
sf_nearest= (spark.table(f"{SILVER}.dist_matrix_step_free").groupBy("block_id").agg(F.min("walk_m").alias("walk_m")))
sf_walks= blocks.join(sf_nearest, "block_id", "left").select("block_id", "est_seniors", "tier_now", "walk_m")
t4, without= tier_counts(sf_walks, LIMIT_MIN)
senior_reach= reach_m(LIMIT_MIN, PACES)[0]  # 900 m
# cut off by steps: a senior can walk to a centre in 15 min on the full network, but not step-free
cut= sf_walks.filter((F.col("tier_now") == "T1") & (F.col("walk_m").isNull() | (F.col("walk_m") > senior_reach)))
results["step_free"]= log_run("step_free_senior_15min", {"network": "step_free", "limit_min": LIMIT_MIN, "pace": PACES["senior"]}, {
    "seniors_cut_off": float(cut.agg(F.sum("est_seniors")).first()[0] or 0),
    "blocks_cut_off": cut.count(),
    "seniors_without_walk": without,
    "t4_blocks": t4,
    "centres_excluded": centres_excluded(f"{SILVER}.dist_matrix_step_free"),
})

In [0]:
# the base-limit run must agree with what the app shows (gold.access_sensitivity)
base= results[LIMIT_MIN]
assert base["seniors_cut_off_total"] > 0, "no seniors cut off at the base limit: gold.access_sensitivity looks empty"
if results["step_free"]["seniors_without_walk"] < base["seniors_without_walk"] - 1:
    print("WARNING: fewer seniors lack a walk step-free than with steps; check the step-free snapping in 03_dist_matrix")
import json
dbutils.notebook.exit(json.dumps({"experiment": EXPERIMENT, "job_run_id": JOB_RUN_ID, **{str(k): v for k, v in results.items()}}))